In [1]:
import logging
from cmes import CMESWrapper
from cmes_run import CMES_Runner

logging.basicConfig(level=logging.INFO)

# Real

### ASSIST

In [2]:
runner = CMES_Runner(
    model_class=CMESWrapper,  
    input_dir='../Data/data/A123',
    split_type="real",
    train_fraction=1,
    epochs=30, 
    seed=123,
    batch_size=256,
    lr=0.002,
    n_clusters=50, 
    topk=20     
)

runner.run()

2026-02-04 11:55:53,621 - INFO - Using device: cuda
2026-02-04 11:55:53,671 - INFO - Loading metadata and p2k mapping...
2026-02-04 11:55:53,722 - INFO - 2493 users, 17676 problems, 123 skills.
2026-02-04 11:55:53,736 - INFO - Pre-building global problem Q-matrix...
2026-02-04 11:55:54,549 - INFO - 
==================== Starting Real-Scenario (Time-Series) Split ====================
2026-02-04 11:55:55,331 - INFO - Building user interaction index...
2026-02-04 11:55:55,385 - INFO - Performing User Clustering (n_clusters=50)...
2026-02-04 11:55:55,937 - INFO - Clustering completed.
2026-02-04 11:55:55,943 - INFO - Pre-calculating sampling pools for each user... This may take a minute.
2026-02-04 11:56:02,107 - INFO - Sampling pools ready.
Epoch 1/30: 100%|██████████| 624/624 [01:23<00:00,  7.44batch/s, loss=2.3664] 
2026-02-04 11:57:32,761 - INFO - Epoch 1 Val Result - AUC: 0.6406, ACC: 0.6650, RMSE: 0.4946
2026-02-04 11:57:32,761 - INFO - Significant improvement detected (0.6406 >= 0.0


==================== Real-Scenario (Time-Series) Split Final Results ====================
Parameters: train_fraction=1, epochs=30, seed=123
AUC:  0.731689
ACC:  0.693051
RMSE: 0.453723
F1:   0.761743


### Junyi

In [2]:
import optuna

def objective(trial):
    n_clusters = trial.suggest_categorical('n_clusters', [50, 100, 150, 200])
    topk = trial.suggest_categorical('topk', [5, 10, 15])

    runner = CMES_Runner(
        model_class=CMESWrapper,  
        input_dir='../Data/data/J123',
        split_type="random",
        train_fraction=1.0, 
        epochs=3,         
        seed=123,
        batch_size=256,     
        lr=0.002,              
        n_clusters=n_clusters, 
        topk=topk              
    )

    val_auc = runner.run_single_fold() 

    return val_auc

study = optuna.create_study(direction='maximize') 
study.optimize(objective, n_trials=10) 

print("Best trial:")
trial = study.best_trial
print(f"  Value: {trial.value}")
print("  Params: ")
for key, value in trial.params.items():
    print(f" {key}: {value} ")

[I 2026-02-04 12:38:32,291] A new study created in memory with name: no-name-e0c3aeb1-af3d-4cc4-8e9a-d3215cafeafd
2026-02-04 12:38:32,336 - INFO - Using device: cuda
2026-02-04 12:38:32,352 - INFO - Loading metadata and p2k mapping...
2026-02-04 12:38:32,355 - INFO - 10000 users, 675 problems, 510 skills.
2026-02-04 12:38:32,357 - INFO - Pre-building global problem Q-matrix...
2026-02-04 12:38:32,394 - INFO - 
==================== Starting Optuna Trial on Fold 1 ====================
2026-02-04 12:38:36,027 - INFO - Building user interaction index...
2026-02-04 12:38:36,148 - INFO - Performing User Clustering (n_clusters=50)...
2026-02-04 12:38:47,991 - INFO - Clustering completed.
2026-02-04 12:38:47,997 - INFO - Pre-calculating sampling pools for each user... This may take a minute.
2026-02-04 12:38:57,833 - INFO - Sampling pools ready.
Epoch 1/3: 100%|██████████| 949/949 [01:26<00:00, 10.95batch/s, loss=1.6355] 
2026-02-04 12:40:38,831 - INFO - Epoch 1 Val Result - AUC: 0.7860, ACC: 

Best trial:
  Value: 0.810751139106343
  Params: 
 n_clusters: 200 
 topk: 5 


In [3]:
runner = CMES_Runner(
    model_class=CMESWrapper,  
    input_dir='../Data/data/J123',
    split_type="real",
    train_fraction=1,
    epochs=30, 
    seed=123,
    batch_size=256,
    lr=0.002,
    n_clusters=200, 
    topk=5     
)

runner.run()

2026-02-04 13:34:25,825 - INFO - Using device: cuda
2026-02-04 13:34:25,829 - INFO - Loading metadata and p2k mapping...
2026-02-04 13:34:25,835 - INFO - 10000 users, 675 problems, 510 skills.
2026-02-04 13:34:25,838 - INFO - Pre-building global problem Q-matrix...
2026-02-04 13:34:25,869 - INFO - 
==================== Starting Real-Scenario (Time-Series) Split ====================
2026-02-04 13:34:29,124 - INFO - Building user interaction index...
2026-02-04 13:34:29,231 - INFO - Performing User Clustering (n_clusters=200)...
2026-02-04 13:34:40,639 - INFO - Clustering completed.
2026-02-04 13:34:40,639 - INFO - Pre-calculating sampling pools for each user... This may take a minute.
2026-02-04 13:35:17,241 - INFO - Sampling pools ready.
Epoch 1/30: 100%|██████████| 945/945 [00:56<00:00, 16.73batch/s, loss=1.2215] 
2026-02-04 13:36:25,781 - INFO - Epoch 1 Val Result - AUC: 0.7710, ACC: 0.7266, RMSE: 0.4333
2026-02-04 13:36:25,781 - INFO - Significant improvement detected (0.7710 >= 0.0


==================== Real-Scenario (Time-Series) Split Final Results ====================
Parameters: train_fraction=1, epochs=30, seed=123
AUC:  0.785866
ACC:  0.732765
RMSE: 0.426147
F1:   0.794101


### NIPS

In [4]:
import optuna

def objective(trial):
    n_clusters = trial.suggest_categorical('n_clusters', [100, 200, 400])
    topk = trial.suggest_categorical('topk', [10, 20, 30])

    runner = CMES_Runner(
        model_class=CMESWrapper,  
        input_dir='../Data/data/J123',
        split_type="random",
        train_fraction=1.0, 
        epochs=3,         
        seed=123,
        batch_size=256,     
        lr=0.002,              
        n_clusters=n_clusters, 
        topk=topk              
    )

    val_auc = runner.run_single_fold() 

    return val_auc

study = optuna.create_study(direction='maximize') 
study.optimize(objective, n_trials=10) 

print("Best trial:")
trial = study.best_trial
print(f"  Value: {trial.value}")
print("  Params: ")
for key, value in trial.params.items():
    print(f" {key}: {value} ")

[I 2026-02-04 13:45:09,441] A new study created in memory with name: no-name-a671fe34-b063-47b7-b409-a32456d8e1b7
2026-02-04 13:45:09,443 - INFO - Using device: cuda
2026-02-04 13:45:09,451 - INFO - Loading metadata and p2k mapping...
2026-02-04 13:45:09,454 - INFO - 10000 users, 675 problems, 510 skills.
2026-02-04 13:45:09,459 - INFO - Pre-building global problem Q-matrix...
2026-02-04 13:45:09,492 - INFO - 
==================== Starting Optuna Trial on Fold 1 ====================
2026-02-04 13:45:13,104 - INFO - Building user interaction index...
2026-02-04 13:45:13,328 - INFO - Performing User Clustering (n_clusters=100)...
2026-02-04 13:45:25,479 - INFO - Clustering completed.
2026-02-04 13:45:25,481 - INFO - Pre-calculating sampling pools for each user... This may take a minute.
2026-02-04 13:45:47,835 - INFO - Sampling pools ready.
Epoch 1/3: 100%|██████████| 949/949 [02:26<00:00,  6.49batch/s, loss=2.5173] 
2026-02-04 13:48:26,156 - INFO - Epoch 1 Val Result - AUC: 0.5000, ACC:

Best trial:
  Value: 0.8104517536694738
  Params: 
 n_clusters: 200 
 topk: 10 


In [5]:
runner = CMES_Runner(
    model_class=CMESWrapper,  
    input_dir='../Data/data/N123',
    split_type="real",
    train_fraction=1,
    epochs=30, 
    seed=123,
    batch_size=256,
    lr=0.002,
    n_clusters=200, 
    topk=10     
)

runner.run()

2026-02-04 15:14:00,418 - INFO - Using device: cuda
2026-02-04 15:14:00,420 - INFO - Loading metadata and p2k mapping...
2026-02-04 15:14:00,441 - INFO - 10000 users, 27599 problems, 388 skills.
2026-02-04 15:14:00,461 - INFO - Pre-building global problem Q-matrix...
2026-02-04 15:14:01,824 - INFO - 
==================== Starting Real-Scenario (Time-Series) Split ====================
2026-02-04 15:14:15,253 - INFO - Building user interaction index...
2026-02-04 15:14:15,530 - INFO - Performing User Clustering (n_clusters=200)...
2026-02-04 15:14:28,398 - INFO - Clustering completed.
2026-02-04 15:14:28,416 - INFO - Pre-calculating sampling pools for each user... This may take a minute.
2026-02-04 15:17:52,501 - INFO - Sampling pools ready.
Epoch 1/30: 100%|██████████| 3906/3906 [10:06<00:00,  6.44batch/s, loss=1.1903] 
2026-02-04 15:28:43,101 - INFO - Epoch 1 Val Result - AUC: 0.7721, ACC: 0.7212, RMSE: 0.4284
2026-02-04 15:28:43,104 - INFO - Significant improvement detected (0.7721 >=


==================== Real-Scenario (Time-Series) Split Final Results ====================
Parameters: train_fraction=1, epochs=30, seed=123
AUC:  0.784480
ACC:  0.729810
RMSE: 0.423076
F1:   0.794950


# Weak

### ASSIST

In [10]:
runner = CMES_Runner(
    model_class=CMESWrapper,  
    input_dir='../Data/data/A123',
    split_type="weak",
    train_fraction=1,
    epochs=30, 
    seed=123,
    batch_size=256,
    lr=0.002,
    n_clusters=50, 
    topk=20     
)

runner.run()

2026-02-05 01:23:45,202 - INFO - Using device: cuda
2026-02-05 01:23:45,221 - INFO - Loading metadata and p2k mapping...
2026-02-05 01:23:45,283 - INFO - 2493 users, 17676 problems, 123 skills.
2026-02-05 01:23:45,353 - INFO - Pre-building global problem Q-matrix...
2026-02-05 01:23:46,667 - INFO - 
==================== Starting Weak-Coverage Split ====================
2026-02-05 01:23:47,617 - INFO - Building user interaction index...
2026-02-05 01:23:47,685 - INFO - Performing User Clustering (n_clusters=50)...
2026-02-05 01:23:48,304 - INFO - Clustering completed.
2026-02-05 01:23:48,315 - INFO - Pre-calculating sampling pools for each user... This may take a minute.
2026-02-05 01:23:59,139 - INFO - Sampling pools ready.
Epoch 1/30: 100%|██████████| 630/630 [01:29<00:00,  7.07batch/s, loss=2.4434] 
2026-02-05 01:25:33,711 - INFO - Epoch 1 Val Result - AUC: 0.6550, ACC: 0.6475, RMSE: 0.4967
2026-02-05 01:25:33,711 - INFO - Significant improvement detected (0.6550 >= 0.0000 + 0.001)
2


==================== Weak-Coverage Split Final Results ====================
Parameters: train_fraction=1, epochs=30, seed=123
AUC:  0.714757
ACC:  0.704542
RMSE: 0.445786
F1:   0.789398


### Junyi

In [8]:
runner = CMES_Runner(
    model_class=CMESWrapper,  
    input_dir='../Data/data/J123',
    split_type="weak",
    train_fraction=1,
    epochs=30, 
    seed=123,
    batch_size=256,
    lr=0.002,
    n_clusters=200, 
    topk=5     
)

runner.run()

2026-02-05 00:05:34,540 - INFO - Using device: cuda
2026-02-05 00:05:34,561 - INFO - Loading metadata and p2k mapping...
2026-02-05 00:05:34,569 - INFO - 10000 users, 675 problems, 510 skills.
2026-02-05 00:05:34,617 - INFO - Pre-building global problem Q-matrix...
2026-02-05 00:05:34,924 - INFO - 
==================== Starting Weak-Coverage Split ====================
2026-02-05 00:05:39,211 - INFO - Building user interaction index...
2026-02-05 00:05:39,359 - INFO - Performing User Clustering (n_clusters=200)...
2026-02-05 00:05:52,322 - INFO - Clustering completed.
2026-02-05 00:05:52,326 - INFO - Pre-calculating sampling pools for each user... This may take a minute.
2026-02-05 00:06:52,555 - INFO - Sampling pools ready.
Epoch 1/30: 100%|██████████| 972/972 [01:02<00:00, 15.64batch/s, loss=1.2436] 
2026-02-05 00:08:10,920 - INFO - Epoch 1 Val Result - AUC: 0.7653, ACC: 0.6794, RMSE: 0.4700
2026-02-05 00:08:10,921 - INFO - Significant improvement detected (0.7653 >= 0.0000 + 0.001)
2


==================== Weak-Coverage Split Final Results ====================
Parameters: train_fraction=1, epochs=30, seed=123
AUC:  0.747579
ACC:  0.699539
RMSE: 0.444062
F1:   0.766003


### NIPS

In [9]:
runner = CMES_Runner(
    model_class=CMESWrapper,  
    input_dir='../Data/data/N123',
    split_type="weak",
    train_fraction=1,
    epochs=30, 
    seed=123,
    batch_size=256,
    lr=0.002,
    n_clusters=200, 
    topk=10     
)

runner.run()

2026-02-05 00:15:04,882 - INFO - Using device: cuda
2026-02-05 00:15:04,885 - INFO - Loading metadata and p2k mapping...
2026-02-05 00:15:04,900 - INFO - 10000 users, 27599 problems, 388 skills.
2026-02-05 00:15:04,915 - INFO - Pre-building global problem Q-matrix...
2026-02-05 00:15:06,403 - INFO - 
==================== Starting Weak-Coverage Split ====================
2026-02-05 00:15:26,124 - INFO - Building user interaction index...
2026-02-05 00:15:26,658 - INFO - Performing User Clustering (n_clusters=200)...
2026-02-05 00:15:37,977 - INFO - Clustering completed.
2026-02-05 00:15:37,989 - INFO - Pre-calculating sampling pools for each user... This may take a minute.
2026-02-05 00:19:30,276 - INFO - Sampling pools ready.
Epoch 1/30: 100%|██████████| 3929/3929 [10:24<00:00,  6.29batch/s, loss=1.2597] 
2026-02-05 00:30:42,814 - INFO - Epoch 1 Val Result - AUC: 0.7763, ACC: 0.7260, RMSE: 0.4252
2026-02-05 00:30:42,814 - INFO - Significant improvement detected (0.7763 >= 0.0000 + 0.00


==================== Weak-Coverage Split Final Results ====================
Parameters: train_fraction=1, epochs=30, seed=123
AUC:  0.754010
ACC:  0.700650
RMSE: 0.442631
F1:   0.756467


# Random

### ASSIST

In [11]:
runner = CMES_Runner(
    model_class=CMESWrapper,  
    input_dir='../Data/data/A123',
    split_type="random",
    train_fraction=1,
    epochs=30, 
    seed=123,
    batch_size=256,
    lr=0.002,
    n_clusters=50, 
    topk=20     
)

runner.run()

2026-02-05 01:38:01,436 - INFO - Using device: cuda
2026-02-05 01:38:01,440 - INFO - Loading metadata and p2k mapping...
2026-02-05 01:38:01,449 - INFO - 2493 users, 17676 problems, 123 skills.
2026-02-05 01:38:01,462 - INFO - Pre-building global problem Q-matrix...
2026-02-05 01:38:02,383 - INFO - 
==================== Starting Random Split Fold 1/5 ====================
2026-02-05 01:38:03,159 - INFO - Building user interaction index...
2026-02-05 01:38:03,212 - INFO - Performing User Clustering (n_clusters=50)...
2026-02-05 01:38:03,784 - INFO - Clustering completed.
2026-02-05 01:38:03,788 - INFO - Pre-calculating sampling pools for each user... This may take a minute.
2026-02-05 01:38:14,024 - INFO - Sampling pools ready.
Epoch 1/30: 100%|██████████| 624/624 [01:27<00:00,  7.10batch/s, loss=2.4691] 
2026-02-05 01:39:46,543 - INFO - Epoch 1 Val Result - AUC: 0.6526, ACC: 0.6574, RMSE: 0.4870
2026-02-05 01:39:46,543 - INFO - Significant improvement detected (0.6526 >= 0.0000 + 0.001)


==================== 5-Fold Cross-Validation Summary (Random Split) ====================
Parameters: train_fraction=1, epochs=30, seed=123
Average AUC:  0.757493 ± 0.001738
Average ACC:  0.725467 ± 0.001617
Average RMSE: 0.431405 ± 0.000664
Average F1:   0.803360 ± 0.002734


### Junyi

In [6]:
runner = CMES_Runner(
    model_class=CMESWrapper,  
    input_dir='../Data/data/J123',
    split_type="random",
    train_fraction=1,
    epochs=30, 
    seed=123,
    batch_size=256,
    lr=0.002,
    n_clusters=200, 
    topk=5     
)

runner.run()

2026-02-04 17:37:19,506 - INFO - Using device: cuda
2026-02-04 17:37:19,527 - INFO - Loading metadata and p2k mapping...
2026-02-04 17:37:19,538 - INFO - 10000 users, 675 problems, 510 skills.
2026-02-04 17:37:19,582 - INFO - Pre-building global problem Q-matrix...
2026-02-04 17:37:19,811 - INFO - 
==================== Starting Random Split Fold 1/5 ====================
2026-02-04 17:37:23,611 - INFO - Building user interaction index...
2026-02-04 17:37:23,758 - INFO - Performing User Clustering (n_clusters=200)...
2026-02-04 17:37:35,957 - INFO - Clustering completed.
2026-02-04 17:37:35,961 - INFO - Pre-calculating sampling pools for each user... This may take a minute.
2026-02-04 17:38:18,617 - INFO - Sampling pools ready.
Epoch 1/30: 100%|██████████| 949/949 [00:58<00:00, 16.35batch/s, loss=1.3999] 
2026-02-04 17:39:29,443 - INFO - Epoch 1 Val Result - AUC: 0.7738, ACC: 0.7115, RMSE: 0.4550
2026-02-04 17:39:29,443 - INFO - Significant improvement detected (0.7738 >= 0.0000 + 0.001)


==================== 5-Fold Cross-Validation Summary (Random Split) ====================
Parameters: train_fraction=1, epochs=30, seed=123
Average AUC:  0.813454 ± 0.001202
Average ACC:  0.766874 ± 0.004508
Average RMSE: 0.399223 ± 0.002999
Average F1:   0.832870 ± 0.007753


### NIPS

In [7]:
runner = CMES_Runner(
    model_class=CMESWrapper,  
    input_dir='../Data/data/N123',
    split_type="random",
    train_fraction=1,
    epochs=30, 
    seed=123,
    batch_size=256,
    lr=0.002,
    n_clusters=200, 
    topk=10     
)

runner.run()

2026-02-04 18:19:20,554 - INFO - Using device: cuda
2026-02-04 18:19:20,564 - INFO - Loading metadata and p2k mapping...
2026-02-04 18:19:20,577 - INFO - 10000 users, 27599 problems, 388 skills.
2026-02-04 18:19:20,590 - INFO - Pre-building global problem Q-matrix...
2026-02-04 18:19:22,065 - INFO - 
==================== Starting Random Split Fold 1/5 ====================
2026-02-04 18:19:35,691 - INFO - Building user interaction index...
2026-02-04 18:19:36,080 - INFO - Performing User Clustering (n_clusters=200)...
2026-02-04 18:19:46,711 - INFO - Clustering completed.
2026-02-04 18:19:46,721 - INFO - Pre-calculating sampling pools for each user... This may take a minute.
2026-02-04 18:22:59,133 - INFO - Sampling pools ready.
Epoch 1/30: 100%|██████████| 3909/3909 [09:36<00:00,  6.78batch/s, loss=1.4042] 
2026-02-04 18:33:13,996 - INFO - Epoch 1 Val Result - AUC: 0.7729, ACC: 0.7207, RMSE: 0.4278
2026-02-04 18:33:13,998 - INFO - Significant improvement detected (0.7729 >= 0.0000 + 0.


==================== 5-Fold Cross-Validation Summary (Random Split) ====================
Parameters: train_fraction=1, epochs=30, seed=123
Average AUC:  0.783275 ± 0.000770
Average ACC:  0.727325 ± 0.001603
Average RMSE: 0.425003 ± 0.000968
Average F1:   0.791421 ± 0.005436
